# Supplementary — Method Contrast (Ensembl ↔ CAT)

Balanced views of RBH coverage, transcript concordance, and CDS classifications.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

OUTPUT_DIR  = Path('../results')
QC_DIR      = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUMMARY_DIR = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR  = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)
RBH_COVERAGE_THRESHOLD = float(globals().get('RBH_COVERAGE_THRESHOLD', 0.95))
SAMPLE_POINTS = int(globals().get('SAMPLE_POINTS', 20000))


In [ ]:
# Panel A — RBH coverage scatter (sampled)
rbh_files = sorted(RESULTS_DIR.rglob('*.gene_pairs_rbh.tsv'))
rows = []
for rbh_path in rbh_files:
    try:
        df = pd.read_csv(rbh_path, sep='	', usecols=['frac_ensembl_covered','frac_cat_covered'])
    except Exception:
        continue
    if len(df) == 0:
        continue
    if SAMPLE_POINTS and len(df) > SAMPLE_POINTS//max(len(rbh_files),1):
        df = df.sample(SAMPLE_POINTS//max(len(rbh_files),1), random_state=42)
    rows.append(df)
rbh_sample = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['frac_ensembl_covered','frac_cat_covered'])

# Panel B — Transcript-concordance distributions (two violins)
tc_rates = []
for p in QC_DIR.rglob('*_transcript_concordance.tsv'):
    try:
        df = pd.read_csv(p, sep='	', usecols=['ens_to_cat_concordance_rate','cat_to_ens_concordance_rate'])
        tc_rates.append(df)
    except Exception:
        continue

rates = pd.concat(tc_rates, ignore_index=True) if tc_rates else pd.DataFrame({'ens_to_cat_concordance_rate':[], 'cat_to_ens_concordance_rate':[]})

# Panel C — CDS classification histogram
cls_counts = []
for p in QC_DIR.rglob('*_coding_integrity.tsv'):
    try:
        df = pd.read_csv(p, sep='	', usecols=['classification'])
        cls_counts.append(df)
    except Exception:
        continue
cls = pd.concat(cls_counts, ignore_index=True) if cls_counts else pd.DataFrame({'classification':[]})

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.2))

# A — coverage scatter/hexbin
ax = axes[0]
if len(rbh_sample):
    hb = ax.hexbin(rbh_sample['frac_ensembl_covered']*100, rbh_sample['frac_cat_covered']*100, gridsize=35, cmap='viridis', mincnt=1)
    ax.axvline(RBH_COVERAGE_THRESHOLD*100, color='red', lw=0.8, ls='--', alpha=0.6)
    ax.axhline(RBH_COVERAGE_THRESHOLD*100, color='red', lw=0.8, ls='--', alpha=0.6)
    ax.set_xlabel('Ensembl gene-body coverage (%)')
    ax.set_ylabel('CAT gene-body coverage (%)')
    ax.set_title('A: RBH coverage (sampled)')
else:
    ax.text(0.5, 0.5, 'No RBH data', transform=ax.transAxes, ha='center')

# B — violins of directional concordance
ax = axes[1]
if len(rates):
    data = [rates['ens_to_cat_concordance_rate']*100, rates['cat_to_ens_concordance_rate']*100]
    ax.violinplot(data, showmedians=True)
    ax.set_xticks([1,2])
    ax.set_xticklabels(['Ensembl→CAT','CAT→Ensembl'], rotation=0)
    ax.set_ylabel('Transcript exact-match rate (%)')
    ax.set_title('B: Transcript concordance')
else:
    ax.text(0.5, 0.5, 'No transcript concordance data', transform=ax.transAxes, ha='center')

# C — classification histogram
ax = axes[2]
if len(cls):
    top = cls['classification'].value_counts().head(8)
    ax.barh(top.index, top.values, color='#8da0cb')
    ax.invert_yaxis()
    ax.set_title('C: CDS classification (top 8)')
else:
    ax.text(0.5, 0.5, 'No coding integrity data', transform=ax.transAxes, ha='center')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout(w_pad=2.0)
fig.savefig(FIGURE_DIR / 'figure_method_contrast.svg', bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'figure_method_contrast.png', dpi=300, bbox_inches='tight')
print('Saved method contrast figure')
